# 05 — EDA II: Análise Estatística das Features

**Objetivo:** avaliar qualidade e potencial preditivo das variáveis.

**Análises:**
* Missing values e cardinalidade
* Outliers e distribuições
* Correlação de Pearson e Spearman
* **Information Value (IV)** / WOE por variável — *calculado somente no TREINO*
* **PSI** entre períodos (estabilidade temporal) e investigação de data leakage

> ⚠️ Ajustes supervisionados (bins por target, WOE) são aprendidos **somente no treino** — nunca sobre toda a base, senão o IV fica otimista e há vazamento do target.

In [0]:
# ── Configuração central (idem notebook 00) ──
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"
TRAIN_END = "2024-05-12"   # snapshots de treino (validação: 13–14/05; teste OOT: 15/05)

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

features  = spark.table(f"{CATALOG}.{SCHEMA}.customer_features")
target_tb = spark.table(f"{CATALOG}.{SCHEMA}.population_target")
df_model  = features.join(target_tb, on=["reference_date", "customerID"], how="inner")

feat_num = [
    "recencia_dias", "freq_compras_5d", "valor_total_5d", "ticket_medio_5d",
    "qtde_itens_5d", "produtos_distintos_5d", "franquias_distintas_5d",
    "preco_medio_5d", "tendencia_2d_vs_3a5",
]

# Base em pandas (≈ 3.5 mil linhas — cabe em memória com segurança)
pdf = df_model.select("reference_date", "customerID", *feat_num, "pagamento_predominante", "target").toPandas()
pdf_train = pdf[pdf["reference_date"] <= pd.Timestamp(TRAIN_END).date()]
print(f"Total: {len(pdf)} linhas | Treino: {len(pdf_train)} linhas")

In [0]:
# 1) Missing values e cardinalidade
qualidade = pd.DataFrame({
    "feature": feat_num + ["pagamento_predominante"],
    "missing_%": [round(pdf[c].isna().mean() * 100, 2) for c in feat_num + ["pagamento_predominante"]],
    "cardinalidade": [pdf[c].nunique() for c in feat_num + ["pagamento_predominante"]],
})
display(qualidade)

# 2) Distribuições e outliers (percentis extremos)
display(pdf[feat_num].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

In [0]:
# 3) Correlação de Pearson e Spearman (multicolinearidade)
corr_pearson = pdf_train[feat_num].corr("pearson")
corr_spearman = pdf_train[feat_num].corr("spearman")
display(corr_pearson.round(2))
display(corr_spearman.round(2))

# Pares com |corr| > 0.8 (candidatos a remoção por colinearidade)
pares = (
    corr_pearson.where(np.triu(np.ones(corr_pearson.shape), k=1).astype(bool))
    .stack().sort_values(key=abs, ascending=False)
)
print("Pares com |corr| > 0.8:")
print(pares[abs(pares) > 0.8] if (abs(pares) > 0.8).any() else "nenhum")

In [0]:
# 4) Information Value (IV) por variável — SOMENTE TREINO
def information_value(train_pdf, col, target="target", bins=10):
    """IV com bins por quantil. Referência de leitura:
    <0.02 inútil | 0.02–0.1 fraco | 0.1–0.3 médio | >0.3 forte (Siddiqi)."""
    tmp = train_pdf[[col, target]].dropna().copy()
    try:
        tmp["bin"] = pd.qcut(tmp[col].rank(method="first"), q=bins, duplicates="drop")
    except ValueError:
        return np.nan
    g = tmp.groupby("bin", observed=True)[target].agg(["count", "sum"])
    g["nao_eventos"] = g["count"] - g["sum"]
    g["pct_ev"]  = g["sum"] / g["sum"].sum()
    g["pct_nev"] = g["nao_eventos"] / g["nao_eventos"].sum()
    return float(((g["pct_ev"] - g["pct_nev"]) * np.log((g["pct_ev"] + 1e-6) / (g["pct_nev"] + 1e-6))).sum())

ivs = pd.DataFrame({
    "feature": feat_num,
    "IV_treino": [round(information_value(pdf_train, c), 4) for c in feat_num],
}).sort_values("IV_treino", ascending=False)
display(ivs)

# WOE de exemplo: recência em bins
bins_rec = pd.cut(pdf_train["recencia_dias"], bins=[0, 1, 2, 3, 4, 5], right=True)
woe_rec = pdf_train.groupby(bins_rec, observed=True)["target"].agg(["count", "mean"])
woe_rec.columns = ["clientes", "taxa_recompra"]
display(woe_rec)

In [0]:
# 5) PSI — estabilidade temporal das variáveis (primeiro vs. último snapshot)
def psi_var(pdf, col, bins=10):
    """PSI < 0.1 estável | 0.1–0.25 atenção | > 0.25 instável."""
    datas = sorted(pdf["reference_date"].unique())
    base, recente = pdf[pdf["reference_date"] == datas[0]][col], pdf[pdf["reference_date"] == datas[-1]][col]
    cortes = np.unique(np.quantile(pdf[col], np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return 0.0
    cortes[0], cortes[-1] = -np.inf, np.inf
    e = np.histogram(base, cortes)[0] / len(base)
    a = np.histogram(recente, cortes)[0] / len(recente)
    e, a = e + 1e-6, a + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

psis = pd.DataFrame({
    "feature": feat_num,
    "PSI": [round(psi_var(pdf, c), 4) for c in feat_num],
}).sort_values("PSI", ascending=False)
display(psis)

In [0]:
# 6) Investigação de data leakage: correlação suspeita com o target
# Features com |corr com target| muito alta (> 0.9) merecem auditoria manual
leak = pdf_train[feat_num + ["target"]].corr()["target"].drop("target").round(3).sort_values(key=abs, ascending=False)
display(leak.to_frame("corr_com_target"))

# Consolidar análise e salvar feature_analysis
analise = ivs.merge(psis, on="feature", how="outer")
analise["status"] = np.where(analise["PSI"] > 0.25, "instavel", np.where(analise["IV_treino"] < 0.02, "fraca", "candidata"))
(spark.createDataFrame(analise)
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.feature_analysis"))
display(analise.sort_values("IV_treino", ascending=False))

## Conclusões

* `feature_analysis` consolida IV (poder preditivo, medido no treino) e PSI (estabilidade temporal).
* Variáveis com **IV < 0.02** são candidatas à exclusão; **PSI > 0.25** indica instabilidade entre períodos.
* Expectativa realista: em dados sintéticos os IVs tendem a ser baixos — isso **não invalida o estudo**, apenas limita o AUC achievable.
* **Próximo passo:** `06_feature_engineering` — transformações aprendidas no treino e bases train/validation/test.